# 📘 Capítulo 4: Aplicaciones de la Derivada a los Negocios
### Métodos Cuantitativos para la Economía — MSc. Jeel Cueva

**Contenido:** Maximización de beneficio (IM=CM), elasticidad puntual, costo medio, extremos absolutos, monotonía, concavidad, puntos de inflexión, TVM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.optimize import fsolve, brentq

plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.3,
    'axes.spines.top':False,'axes.spines.right':False})
C_AZUL='#1f77b4'; C_ROJO='#d62728'; C_VERDE='#2ca02c'
C_NARANJA='#ff7f0e'; C_MORADO='#9467bd'
print('Librerías cargadas.')

## Ejemplo 4.1 — Maximización del Beneficio: IM = CM
### $p(x)=200-x$, $C(x)=x^3-20x^2+120x+500$

In [ ]:
# EJEMPLO 4.1 — Maximización del beneficio: IM = CM
x = sp.Symbol('x', positive=True)
p_fun = 200 - x
I_fun = p_fun*x; C_fun = x**3-20*x**2+120*x+500
pi_fun = I_fun - C_fun
IM = sp.diff(I_fun, x); CM = sp.diff(C_fun, x)
# Condición CPO: IM = CM
eq = sp.Eq(IM, CM)
soluciones = sp.solve(eq, x)
pi2 = sp.diff(pi_fun, x, 2)
print('MAXIMIZACIÓN DEL BENEFICIO')
for sol in soluciones:
    if sol.is_real and float(sol) > 0:
        p_opt  = float(p_fun.subs(x,sol))
        pi_opt = float(pi_fun.subs(x,sol))
        pi2_v  = float(pi2.subs(x,sol))
        tipo = 'MÁXIMO' if pi2_v < 0 else 'MÍNIMO'
        print(f'  x*={float(sol):.2f}, P*={p_opt:.2f}, π*={pi_opt:.2f}, π"={pi2_v:.2f} → {tipo}')

x_opt_f = float([s for s in soluciones if s.is_real and float(s)>0 and float(pi2.subs(x,s))<0][0])
x_v = np.linspace(0,25,400)
I_v  = (200-x_v)*x_v; C_v = x_v**3-20*x_v**2+120*x_v+500; pi_v = I_v-C_v
IM_v = 200-2*x_v; CM_v = 3*x_v**2-40*x_v+120

fig,axes = plt.subplots(1,2,figsize=(13,5))
ax=axes[0]
ax.plot(x_v,I_v,color=C_VERDE,lw=2.5,label='Ingreso $I(x)$')
ax.plot(x_v,C_v,color=C_ROJO,lw=2.5,label='Costo $C(x)$')
ax.axvline(x_opt_f,color='gray',ls=':',lw=1)
ax.fill_between(x_v,I_v,C_v,where=(I_v>C_v),alpha=0.2,color=C_VERDE,label='Beneficio')
ax.set_xlabel('x'); ax.set_ylabel('Miles de S/'); ax.set_title('I(x) vs C(x)'); ax.legend(fontsize=9)

ax2=axes[1]
ax2.plot(x_v,IM_v,color=C_AZUL,lw=2.5,label='$IM=200-2x$')
ax2.plot(x_v,CM_v,color=C_ROJO,lw=2.5,label='$CM=3x^2-40x+120$')
ax2.axvline(x_opt_f,color='gray',ls=':',lw=1)
ax2.scatter([x_opt_f],[float(IM.subs(x,x_opt_f))],color='black',s=120,zorder=6,label=f'IM=CM en x*≈{x_opt_f:.1f}')
ax2.set_xlabel('x'); ax2.set_ylabel('Marginal'); ax2.set_title('IM = CM → Óptimo'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig_c04_IM_CM.png',bbox_inches='tight'); plt.show()

## Ejemplo 4.2 — Elasticidad y Estrategia de Precios
### $D(P)=500-2P^2$ — relación elasticidad-ingreso

In [ ]:
# EJEMPLO 4.2 — Elasticidad puntual y estrategia de precios
P = sp.Symbol('P', positive=True)
D_fun = 500 - 2*P**2
dD = sp.diff(D_fun, P)
eps = dD*(P/D_fun)
# Precio unitario: |ε|=1
P_unit = sp.solve(sp.Eq(eps, -1), P)[0]
# Ingreso total
IT = P*D_fun
dIT = sp.diff(IT, P)
P_maxIT = sp.solve(dIT, P)[0]
print(f'D(P) = 500 - 2P²')
print(f'ε(P) = {sp.simplify(eps)}')
print(f'|ε|=1 cuando P = {P_unit:.4f}')
print(f'IT máximo en P = {P_maxIT:.4f}')
print(f'Son iguales: {sp.simplify(P_unit - P_maxIT) == 0}')

# Evaluaciones
for Pv in [5,8,10,12,15]:
    Dv = 500-2*Pv**2
    ev = float(eps.subs(P,Pv))
    ITv= Pv*Dv
    tipo='Elástica' if abs(ev)>1 else ('Unitaria' if abs(abs(ev)-1)<0.05 else 'Inelástica')
    print(f'P={Pv:>3}: D={Dv:>5}, ε={ev:>6.3f} ({tipo}), IT={ITv:>7}')

P_v = np.linspace(1,15,300)
D_v = 500-2*P_v**2
mask = D_v > 0
P_v,D_v = P_v[mask],D_v[mask]
IT_v = P_v*D_v
eps_v = (-4*P_v**2)/(500-2*P_v**2)

fig,axes = plt.subplots(1,3,figsize=(15,5))
ax=axes[0]
ax.plot(D_v,P_v,color=C_AZUL,lw=2.5,label='$D(P)=500-2P^2$')
P_u = float(P_unit)
ax.scatter([float(D_fun.subs(P,P_u))],[P_u],color=C_ROJO,s=100,zorder=6,label='|ε|=1')
ax.set_xlabel('Q'); ax.set_ylabel('P'); ax.set_title('Curva de Demanda'); ax.legend()

ax2=axes[1]
ax2.plot(P_v,IT_v,color=C_VERDE,lw=2.5)
ax2.axvline(float(P_maxIT),color='gray',ls='--',lw=1)
ax2.scatter([float(P_maxIT)],[float(IT.subs(P,P_maxIT))],color=C_ROJO,s=100,zorder=6,label=f'IT máx')
ax2.set_xlabel('P'); ax2.set_ylabel('IT=P·D'); ax2.set_title('Ingreso Total'); ax2.legend()

ax3=axes[2]
ax3.plot(P_v,eps_v,color=C_MORADO,lw=2.5)
ax3.axhline(-1,color='gray',ls='--',lw=1.5,label='|ε|=1')
ax3.fill_between(P_v,eps_v,-1,where=(eps_v<-1),alpha=0.25,color=C_AZUL,label='Elástica')
ax3.fill_between(P_v,eps_v,-1,where=(eps_v>-1),alpha=0.25,color=C_ROJO,label='Inelástica')
ax3.set_xlabel('P'); ax3.set_ylabel('ε(P)'); ax3.set_title('Elasticidad vs Precio'); ax3.legend(fontsize=9)
plt.suptitle('Elasticidad y Estrategia de Precios', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('fig_c04_elasticidad.png',bbox_inches='tight'); plt.show()

## Ejemplo 4.3 — Mínimo del Costo Medio y Análisis Completo
### $C(x)=x^3-6x^2+15x+100$ — análisis gráfico completo

In [ ]:
# EJEMPLO 4.3 — Mínimo del costo medio y análisis completo de π(x)
x = sp.Symbol('x', positive=True)

# Costo medio
C_fun = x**3 - 6*x**2 + 15*x + 100
C_med = C_fun/x
CM_f  = sp.diff(C_fun, x)  # Costo marginal
dCm   = sp.diff(C_med, x)
x_min_cm = sp.solve(dCm, x)
x_min_cm_pos = [float(r) for r in x_min_cm if r.is_real and float(r)>0]
print('MÍNIMO DEL COSTO MEDIO')
for xm in x_min_cm_pos:
    print(f'  x={xm:.4f}: CM(x)={float(CM_f.subs(x,xm)):.4f}, C̄(x)={float(C_med.subs(x,xm)):.4f}')
    print(f'  CM = C̄ en el mínimo: {abs(float(CM_f.subs(x,xm))-float(C_med.subs(x,xm)))<0.01}')

# Análisis completo: π(x) = -x³+12x²-36x+48 en [0,10]
x2 = sp.Symbol('x')
pi = -x2**3 + 12*x2**2 - 36*x2 + 48
pi1 = sp.diff(pi, x2)
pi2 = sp.diff(pi1, x2)
criticos = sp.solve(pi1, x2)
puntos_eval = [0] + [float(c) for c in criticos if 0<float(c)<10] + [10]
print('\nANÁLISIS COMPLETO: π(x)=-x³+12x²-36x+48 en [0,10]')
for xp in puntos_eval:
    piv = float(pi.subs(x2,xp))
    pi2v = float(pi2.subs(x2,xp))
    tipo = '(máx local)' if pi2v<0 else ('(mín local)' if pi2v>0 else '(extremo intervalo)')
    print(f'  x={xp:.1f}: π={piv:.1f}, π"={pi2v:.1f} {tipo}')

x_v = np.linspace(0.5, 12, 400)
C_v = x_v**3-6*x_v**2+15*x_v+100
CM_v = 3*x_v**2-12*x_v+15
Cmed_v = C_v/x_v

x_v2 = np.linspace(0,10,400)
pi_v = -x_v2**3+12*x_v2**2-36*x_v2+48
pi1_v= -3*x_v2**2+24*x_v2-36
pi2_v= -6*x_v2+24

fig,axes = plt.subplots(1,3,figsize=(15,5))
ax=axes[0]
ax.plot(x_v,CM_v,color=C_VERDE,lw=2.5,label='$CM=C\'(x)$')
ax.plot(x_v,Cmed_v,color=C_AZUL,lw=2.5,label='$\\bar{C}=C(x)/x$')
if x_min_cm_pos:
    xm=x_min_cm_pos[0]
    ax.scatter([xm],[float(C_med.subs(x,xm))],color=C_ROJO,s=120,zorder=6,
        label=f'Mínimo x≈{xm:.1f}')
ax.set_xlabel('x'); ax.set_ylim(0,60)
ax.set_title('Costo Marginal = Costo Medio\nen el Mínimo'); ax.legend()

ax2=axes[1]
ax2.plot(x_v2,pi_v,color=C_MORADO,lw=2.5,label='$\\pi(x)$')
ax2.plot(x_v2,pi1_v,color=C_VERDE,lw=2,ls='--',label="$\\pi'(x)$")
ax2.axhline(0,color='black',lw=0.8)
for xp,yp in [(2,float(pi.subs(x2,2))),(6,float(pi.subs(x2,6)))]:
    ax2.scatter([xp],[yp],color=C_ROJO,s=100,zorder=6)
    ax2.annotate(f'({xp},{yp:.0f})',xy=(xp,yp),xytext=(xp+0.3,yp+5),fontsize=9)
ax2.set_xlabel('x'); ax2.set_title('Análisis Completo de π(x)'); ax2.legend(fontsize=9)

ax3=axes[2]
ax3.plot(x_v2,pi2_v,color=C_NARANJA,lw=2.5,label="$\\pi''(x)$")
ax3.axhline(0,color='black',lw=0.8)
ax3.axvline(4,color='gray',ls='--',lw=1,label='Inflexión x=4')
ax3.fill_between(x_v2,pi2_v,0,where=(x_v2<4),alpha=0.25,color=C_VERDE,label='Cóncava ↑')
ax3.fill_between(x_v2,pi2_v,0,where=(x_v2>4),alpha=0.25,color=C_ROJO,label='Cóncava ↓')
ax3.set_xlabel('x'); ax3.set_title('Concavidad: Segunda Derivada'); ax3.legend(fontsize=9)
plt.suptitle('Optimización y Análisis Completo', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('fig_c04_analisis_completo.png',bbox_inches='tight'); plt.show()

## Ejemplo 4.4 — Modelo de Inventarios EOQ
### $CT(Q) = Ds/Q + hQ/2$ — Lote económico

In [ ]:
# EJEMPLO 4.4 — Modelo EOQ de inventarios
Q = sp.Symbol('Q', positive=True)
D_val, s_val, h_val = 1200, 50, 2
CT = D_val*s_val/Q + h_val*Q/2
dCT = sp.diff(CT, Q)
Q_opt = sp.solve(dCT, Q)[0]
CT_opt = float(CT.subs(Q, Q_opt))
d2CT = float(sp.diff(CT,Q,2).subs(Q,Q_opt))
print(f'MODELO EOQ: D={D_val}, s={s_val}, h={h_val}')
print(f'CT(Q) = {D_val*s_val}/Q + {h_val/2}Q')
print(f"CT'(Q) = {dCT}")
print(f'Q* = √(2Ds/h) = √({2*D_val*s_val}/{h_val}) = {float(Q_opt):.2f}')
print(f'CT(Q*) = {CT_opt:.2f}')
print(f"CT''(Q*) = {d2CT:.4f} > 0 → MÍNIMO")
print(f'Pedidos por año: {D_val/float(Q_opt):.1f}')

Q_v = np.linspace(1, 500, 400)
CT_v = D_val*s_val/Q_v + h_val*Q_v/2
CT_pedido = D_val*s_val/Q_v
CT_mantenimiento = h_val*Q_v/2

fig,axes = plt.subplots(1,2,figsize=(13,5))
ax=axes[0]
ax.plot(Q_v,CT_v,color=C_AZUL,lw=2.5,label='CT(Q) total')
ax.plot(Q_v,CT_pedido,color=C_ROJO,lw=1.8,ls='--',label='Costo pedido')
ax.plot(Q_v,CT_mantenimiento,color=C_VERDE,lw=1.8,ls='--',label='Costo mantenimiento')
ax.axvline(float(Q_opt),color='gray',ls=':',lw=1)
ax.scatter([float(Q_opt)],[CT_opt],color='black',s=120,zorder=6,label=f'Q*={float(Q_opt):.0f}')
ax.set_xlabel('Tamaño del lote Q'); ax.set_ylabel('Costo Total')
ax.set_title('Modelo EOQ — Lote Económico'); ax.legend(fontsize=9)

# Análisis de sensibilidad: Q* vs s
s_vals = np.linspace(10,200,100)
Q_opt_s = np.sqrt(2*D_val*s_vals/h_val)
ax2=axes[1]
ax2.plot(s_vals,Q_opt_s,color=C_NARANJA,lw=2.5,label='$Q^*=\\sqrt{2Ds/h}$')
ax2.axvline(s_val,color='gray',ls='--',lw=1,label=f's={s_val} (base)')
ax2.scatter([s_val],[float(Q_opt)],color=C_ROJO,s=100,zorder=6)
ax2.set_xlabel('Costo de pedido s'); ax2.set_ylabel('Q* óptimo')
ax2.set_title('Sensibilidad de Q* al Costo de Pedido'); ax2.legend()
plt.tight_layout(); plt.savefig('fig_c04_EOQ.png',bbox_inches='tight'); plt.show()

## Ejemplo 4.5 — Teorema del Valor Medio (TVM)
### $C(x)=0.5x^2+10x+200$ en $[10,20]$

In [ ]:
# EJEMPLO 4.5 — TVM aplicado a costos
x = sp.Symbol('x')
C = 0.5*x**2 + 10*x + 200
C1 = sp.diff(C, x)  # = x + 10
a_val, b_val = 10, 20
tasa_prom = (float(C.subs(x,b_val)) - float(C.subs(x,a_val))) / (b_val - a_val)
c_val = float(sp.solve(sp.Eq(C1, tasa_prom), x)[0])
print(f'C(x) = 0.5x² + 10x + 200,  C\'(x) = {C1}')
print(f'Tasa promedio [{a_val},{b_val}] = (C({b_val})-C({a_val}))/({b_val}-{a_val}) = {tasa_prom}')
print(f'TVM garantiza c ∈ ({a_val},{b_val}) donde C\'(c) = {tasa_prom}')
print(f'c = {c_val} (punto medio, como esperado para función cuadrática)')

x_v = np.linspace(0, 25, 400)
C_v = 0.5*x_v**2 + 10*x_v + 200
Ca = float(C.subs(x, a_val)); Cb = float(C.subs(x, b_val))
Cc = float(C.subs(x, c_val))

fig,ax = plt.subplots(figsize=(9,6))
ax.plot(x_v,C_v,color=C_AZUL,lw=2.5,label='$C(x)=0.5x^2+10x+200$')
# Secante
m_sec = (Cb-Ca)/(b_val-a_val)
x_sec = np.array([a_val-1,b_val+1])
y_sec = Ca + m_sec*(x_sec-a_val)
ax.plot(x_sec,y_sec,color=C_ROJO,lw=2,ls='--',label=f'Secante (pendiente={m_sec})')
# Tangente en c
x_tan = np.array([c_val-5,c_val+5])
y_tan = Cc + tasa_prom*(x_tan-c_val)
ax.plot(x_tan,y_tan,color=C_VERDE,lw=2,ls='-.',label=f'Tangente en c={c_val} (pendiente={tasa_prom})')
ax.scatter([a_val,b_val],[Ca,Cb],color=C_ROJO,s=80,zorder=6)
ax.scatter([c_val],[Cc],color=C_VERDE,s=100,zorder=6)
ax.annotate(f'c={c_val}',xy=(c_val,Cc),xytext=(c_val+1,Cc-50),fontsize=10,
    arrowprops=dict(arrowstyle='->',color='gray'))
ax.set_xlabel('x'); ax.set_ylabel('C(x)')
ax.set_title('Teorema del Valor Medio\n(Tangente paralela a la secante)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig_c04_TVM.png',bbox_inches='tight'); plt.show()

---
## Resumen del Capítulo 4

| Herramienta | Condición | Interpretación |
|---|---|---|
| Máximo beneficio | $IM = CM$ | Óptimo de producción |
| Máximo IT | $|\\varepsilon|=1$ | Elasticidad unitaria |
| Mínimo costo medio | $CM = \\bar{C}$ | Escala eficiente |
| Máximo/mínimo local | $f'=0$, $f''<0$ / $f''>0$ | CPO + CSO |
| Punto de inflexión | $f''=0$ y cambia signo | Cambio de curvatura |
| TVM | $f'(c)=(f(b)-f(a))/(b-a)$ | Costo marginal promedio |
| EOQ | $Q^*=\\sqrt{2Ds/h}$ | Lote óptimo de inventario |

---
*MSc. Jeel Cueva — UNHEVAL | UDH | UTP — Lima, 2025*